## Setup and Imports

In [28]:
import os
import numpy as np
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification
)
from torch.utils.data import Dataset
import pandas as pd
from tqdm import tqdm
from collections import Counter
from transformers import EarlyStoppingCallback
import json
from pathlib import Path
# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Setup complete")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Setup complete
PyTorch version: 2.11.0.dev20260204+cu128
CUDA available: True


###  Define label space (entity types + BIO tagging)
 definition of the 13 GutBrainIE entity categories and expansions into BIO tags:
 - "O" for tokens outside any entity
 - "B-<label>" for the first token of an entity mention
 - "I-<label>" for continuation tokens
 Then we build `label2id` / `id2label` mappings so the model can train and decode labels.

 Finally we set:
 - the pretrained backbone (BioBERT)
 - the output directory where the fine-tuned model will be saved.

In [29]:
# Define entity labels
ENTITY_LABELS = [
    "anatomical location",
    "animal",
    "bacteria",
    "biomedical technique",
    "chemical",
    "DDF",
    "dietary supplement",
    "drug",
    "food",
    "gene",
    "human",
    "microbiome",
    "statistical technique"
]

# Create BIO tags for each entity label
label_list = ['O']  # Outside
for entity_label in ENTITY_LABELS:
    label_list.append(f'B-{entity_label}')  # Beginning
    label_list.append(f'I-{entity_label}')  # Inside

label2id = {k: v for v, k in enumerate(label_list)}
id2label = {v: k for v, k in enumerate(label_list)}

print(f"Total labels: {len(label_list)}")
print(f"\nFirst 10 labels: {label_list[:10]}")

Total labels: 27

First 10 labels: ['O', 'B-anatomical location', 'I-anatomical location', 'B-animal', 'I-animal', 'B-bacteria', 'I-bacteria', 'B-biomedical technique', 'I-biomedical technique', 'B-chemical']


## Data Loading Functions
This section defines two helper functions:
 - `load_ner_data`: loads multiple JSON annotation files and merges them into a single dictionary keyed by PMID.
 - `prepare_documents_for_ner`: splits each article into two separate training examples:  one for the title and one for the abstract. This is important because entities are
  annotated with a `location` field (title/abstract) and the spans are relative to that text segment.

In [30]:
def prepare_documents_for_ner(data):
    """
    Convert raw data into structured format for NER.
    Each document has title and abstract as separate text segments.
    """
    documents = []
    
    for pmid, article in data.items():
        # Process title
        title_text = article['metadata']['title']
        title_entities = [e for e in article['entities'] if e['location'] == 'title']
        
        documents.append({
            'pmid': pmid,
            'location': 'title',
            'text': title_text,
            'entities': title_entities
        })
        
        # Process abstract
        abstract_text = article['metadata']['abstract']
        abstract_entities = [e for e in article['entities'] if e['location'] == 'abstract']
        
        documents.append({
            'pmid': pmid,
            'location': 'abstract',
            'text': abstract_text,
            'entities': abstract_entities
        })
    
    return documents


print("✓ Data loading functions defined")

✓ Data loading functions defined


## Load Training and Dev Data
loading of the training data (gold/platinum/silver) and development data from the provided dev split. Then it converts articles into per-segment examples (title + abstract), producing `train_documents` and `dev_documents`.

In [31]:
# ----------------------------
PROJECT_ROOT = Path.cwd().parents[1]
DATA_2026 = PROJECT_ROOT / "data" / "GutBrainIE_Full_Collection_2026" / "Annotations"

TRAIN_GOLD   = DATA_2026 / "Train" / "gold_quality"   / "json_format" / "train_gold.json"
TRAIN_SILVER = DATA_2026 / "Train" / "silver_quality" / "json_format" / "train_silver.json"
TRAIN_BRONZE = DATA_2026 / "Train" / "bronze_quality" / "json_format" / "train_bronze.json"  # optional

DEV_PATH = DATA_2026 / "Dev" / "json_format" / "dev.json"

print("TRAIN_GOLD:", TRAIN_GOLD)
print("TRAIN_SILVER:", TRAIN_SILVER)
print("TRAIN_BRONZE:", TRAIN_BRONZE)
print("DEV:", DEV_PATH)


TRAIN_GOLD: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\data\GutBrainIE_Full_Collection_2026\Annotations\Train\gold_quality\json_format\train_gold.json
TRAIN_SILVER: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\data\GutBrainIE_Full_Collection_2026\Annotations\Train\silver_quality\json_format\train_silver.json
TRAIN_BRONZE: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\data\GutBrainIE_Full_Collection_2026\Annotations\Train\bronze_quality\json_format\train_bronze.json
DEV: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\data\GutBrainIE_Full_Collection_2026\Annotations\Dev\json_format\dev.json


In [32]:
def align_labels_with_tokens(text, entities, tokenizer, label2id, max_length=512):
    """
    Create BIO tags for tokenized text based on character-level entity annotations.

    Key fixes vs baseline:
    - Handles inclusive end_idx in dataset by converting to exclusive end for overlap checks.
    - Ignores special tokens and (optionally) can ignore subword-only labeling errors.
    - Deterministic overlap policy: longer spans first; do not overwrite already-labeled tokens.
    - Casts offsets to int for safety.
    """
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
    )

    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]
    offset_mapping = encoding["offset_mapping"]  # list[(start,end)] end is exclusive

    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    # Initialize all labels as 'O'
    labels = ["O"] * len(input_ids)

    # Sort entities: earlier start first, then longer first (so we keep more specific spans)
    sorted_entities = sorted(
        entities,
        key=lambda e: (int(e["start_idx"]), -(int(e["end_idx"]) - int(e["start_idx"]))),
    )

    labeled_positions = set()

    for ent in sorted_entities:
        ent_start = int(ent["start_idx"])
        ent_end_excl = int(ent["end_idx"]) + 1  #  dataset end_idx is inclusive → convert to exclusive
        ent_label = str(ent["label"])

        ent_token_start = None
        ent_token_end = None

        for idx, (tok_start, tok_end) in enumerate(offset_mapping):
            tok_start = int(tok_start)
            tok_end = int(tok_end)

            # Special tokens have (0,0) offsets in HF tokenizers
            if tok_start == 0 and tok_end == 0:
                continue

            # Robust overlap check (character spans)
            if tok_start < ent_end_excl and tok_end > ent_start:
                if ent_token_start is None:
                    ent_token_start = idx
                ent_token_end = idx

        # Apply BIO tags if we found any overlapping tokens
        if ent_token_start is not None and ent_token_end is not None:
            for i in range(ent_token_start, ent_token_end + 1):
                if i in labeled_positions:
                    continue

                if i == ent_token_start:
                    tag = f"B-{ent_label}"
                else:
                    tag = f"I-{ent_label}"

                # Fallback safety: if tag not in label2id, keep O
                if tag in label2id:
                    labels[i] = tag
                    labeled_positions.add(i)

    label_ids = [label2id.get(tag, label2id["O"]) for tag in labels]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": label_ids,
        "tokens": tokens,
    }

In [33]:
class NERDataset(Dataset):
    def __init__(self, processed_data):
        self.data = processed_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(item["labels"], dtype=torch.long),
        }

print("✓ Custom dataset class defined")

✓ Custom dataset class defined


In [34]:
def _load_json(path: Path):
    with path.open(encoding="utf-8") as f:
        return json.load(f)

def safe_load_one(path: Path, name: str):
    if path.exists():
        data = _load_json(path)
        print(f"[OK] {name}: loaded {len(data)} docs from {path.name}")
        return data
    print(f"[SKIP] {name}: missing {path}")
    return {}

def build_processed_segments_from_dict(raw_data_dict, tokenizer, label2id, max_length=512):
    docs = prepare_documents_for_ner(raw_data_dict)
    processed = []
    for doc in tqdm(docs, desc="Align BIO"):
        ex = align_labels_with_tokens(doc["text"], doc["entities"], tokenizer, label2id, max_length=max_length)
        ex["pmid"] = doc["pmid"]
        ex["location"] = doc["location"]
        ex["text"] = doc["text"]
        ex["entities"] = doc["entities"]
        processed.append(ex)
    return processed

def build_dev_dataset(dev_documents, tokenizer, label2id, max_length=512):
    processed_dev = []
    for doc in tqdm(dev_documents, desc="Processing dev"):
        ex = align_labels_with_tokens(doc["text"], doc["entities"], tokenizer, label2id, max_length=max_length)
        ex["pmid"] = doc["pmid"]
        ex["location"] = doc["location"]
        ex["text"] = doc["text"]
        ex["entities"] = doc["entities"]
        processed_dev.append(ex)
    return NERDataset(processed_dev), processed_dev

## Initialize BERT Model and Tokenizer
This cell loads
- the BioBERT tokenizer
 - the BioBERT model with a token-classification head sized to our BIO label space

In [35]:
model_name = "dmis-lab/biobert-v1.1"
output_model_dir = "models/bert_ner_2026_curriculum"

tokenizer = AutoTokenizer.from_pretrained(model_name)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True, return_tensors="pt")

dev_data = _load_json(DEV_PATH)
dev_documents = prepare_documents_for_ner(dev_data)
dev_dataset, processed_dev = build_dev_dataset(dev_documents, tokenizer, label2id, max_length=512)

Processing dev: 100%|██████████| 160/160 [00:00<00:00, 466.48it/s]


# LOAD + PROCESS TRAIN (GOLD/SILVER) 2026

In [36]:
gold_data = safe_load_one(TRAIN_GOLD, "2026 GOLD")
silver_data = safe_load_one(TRAIN_SILVER, "2026 SILVER")

# (Optional) if you want to include Bronze later, keep it commented for now:
# bronze_data = safe_load_one(TRAIN_BRONZE, "2026 BRONZE")

# 2) Build processed segments (title + abstract) with BIO alignment
processed_gold = build_processed_segments_from_dict(
    raw_data_dict=gold_data,
    tokenizer=tokenizer,
    label2id=label2id,
    max_length=512,
)

processed_silver = build_processed_segments_from_dict(
    raw_data_dict=silver_data,
    tokenizer=tokenizer,
    label2id=label2id,
    max_length=512,
)

# 3) Wrap into torch datasets
train_dataset_gold = NERDataset(processed_gold)
train_dataset_silver = NERDataset(processed_silver)

print("\n" + "=" * 60)
print("TRAIN DATASETS READY")
print("=" * 60)
print(f"GOLD docs:   {len(gold_data)}  -> segments: {len(processed_gold)}")
print(f"SILVER docs: {len(silver_data)} -> segments: {len(processed_silver)}")
print(f"train_dataset_gold:   {len(train_dataset_gold)}")
print(f"train_dataset_silver: {len(train_dataset_silver)}")

[OK] 2026 GOLD: loaded 639 docs from train_gold.json
[OK] 2026 SILVER: loaded 811 docs from train_silver.json


Align BIO: 100%|██████████| 1622/1622 [00:04<00:00, 381.33it/s]


TRAIN DATASETS READY
GOLD docs:   639  -> segments: 1278
SILVER docs: 811 -> segments: 1622
train_dataset_gold:   1278
train_dataset_silver: 1622


## Configure Training Arguments

### Define Evaluation metric (seqeval over BIO tags)
 This cell defines a `compute_metrics_seqeval` function for Hugging Face Trainer.
 It converts model outputs (logits) into predicted BIO tags and compares them to gold BIO tags using seqeval.

**Implementation details:**
 - `argmax` selects the most likely tag per token.
 - tokens with label -100 are skipped (ignored padding/special positions).
 - seqeval computes precision/recall/F1 at the entity level from BIO sequences.

In [37]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, EvalPrediction
from seqeval.metrics import precision_score, recall_score, f1_score
import shutil

# ---------- FIX compute_metrics (HF Trainer passes EvalPrediction) ----------
def compute_metrics_seqeval(p: EvalPrediction):
    preds = np.argmax(p.predictions, axis=-1)
    labels = p.label_ids

    true_labels = []
    true_preds = []

    for pred_seq, label_seq in zip(preds, labels):
        seq_true = []
        seq_pred = []
        for p_id, l_id in zip(pred_seq, label_seq):
            if int(l_id) == -100:
                continue
            seq_true.append(id2label[int(l_id)])
            seq_pred.append(id2label[int(p_id)])
        true_labels.append(seq_true)
        true_preds.append(seq_pred)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }


In [38]:
from transformers import TrainingArguments
# ---------- helpers ----------
def is_valid_checkpoint(path: str) -> bool:
    """A HF checkpoint dir is considered valid if it has config + model weights."""
    p = Path(path)
    if not p.exists() or not p.is_dir():
        return False
    has_config = (p / "config.json").exists()
    has_weights = (p / "model.safetensors").exists() or (p / "pytorch_model.bin").exists()
    has_tokenizer = (p / "tokenizer_config.json").exists() or (p / "tokenizer.json").exists()
    return has_config and has_weights and has_tokenizer

def init_model(from_dir_or_name: str):
    return AutoModelForTokenClassification.from_pretrained(
        from_dir_or_name,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
    )

def make_args(output_dir: str, lr: float, epochs: int, warmup_ratio: float, seed: int = 42):
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=lr,
        lr_scheduler_type="linear",
        warmup_ratio=warmup_ratio,
        weight_decay=0.01,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=epochs,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=100,
        save_total_limit=2,
        seed=seed,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )


## Train BERT Model


In [39]:
import torch
from transformers import Trainer

def compute_class_weights(processed_train, num_labels, ignore_index=-100, power=0.5):
    """
    Compute class weights from token label counts.
    power=0.5 -> sqrt inverse frequency (usually stable).
    """
    counts = Counter()
    for ex in processed_train:
        for y in ex["labels"]:
            if y == ignore_index:
                continue
            counts[int(y)] += 1

    # build weights: w_c = (1 / freq_c)^power
    freqs = np.zeros(num_labels, dtype=np.float64)
    for c in range(num_labels):
        freqs[c] = counts.get(c, 0)

    # avoid div-by-zero for unseen classes (shouldn't happen, but safe)
    freqs[freqs == 0] = 1.0

    weights = (1.0 / freqs) ** power

    # normalize weights to mean=1 (keeps loss scale reasonable)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float)

class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits  # (B, T, C)

        # flatten
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None,
            ignore_index=-100
        )
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))

        return (loss, outputs) if return_outputs else loss



# Decide checkpoint paths

In [40]:
ckpt_b1 = output_model_dir + "_B1_gold"
ckpt_b2 = output_model_dir + "_B2_gold_plus_silver"

# Choose which checkpoint to load if available (prefer B2, else B1)
preferred_ckpt = ckpt_b2 if is_valid_checkpoint(ckpt_b2) else (ckpt_b1 if is_valid_checkpoint(ckpt_b1) else None)

# Flag to skip training if checkpoint exists
SKIP_TRAINING_IF_CKPT_EXISTS = False

### STAGE B1: GOLD only + STAGE B2: GOLD oversampled + SILVER

In [41]:
if SKIP_TRAINING_IF_CKPT_EXISTS and preferred_ckpt is not None:
    print("\n" + "=" * 60)
    print(f"[LOAD] Found existing checkpoint -> skipping training: {preferred_ckpt}")
    print("=" * 60)
    final_model_dir = preferred_ckpt

else:
    # -------- STAGE B1: GOLD only --------
    print("\n" + "=" * 60)
    print("STAGE B1: 2026 GOLD (early stopping)")
    print("=" * 60)

    cw_b1 = compute_class_weights(processed_gold, num_labels=len(label_list), power=0.5)
    cw_b1 = torch.clamp(cw_b1, min=0.5, max=5.0)

    args_b1 = make_args(output_dir=ckpt_b1, lr=2e-5, epochs=6, warmup_ratio=0.10)
    model_b1 = init_model(model_name)

    trainer_b1 = WeightedLossTrainer(
        model=model_b1,
        args=args_b1,
        train_dataset=train_dataset_gold,
        eval_dataset=dev_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics_seqeval,
        class_weights=cw_b1,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.0)],
    )

    trainer_b1.train()
    trainer_b1.save_model(ckpt_b1)
    tokenizer.save_pretrained(ckpt_b1)

    # -------- STAGE B2: GOLD*mult + SILVER --------
    print("\n" + "=" * 60)
    GOLD_MULT = 5
    print(f"STAGE B2: (GOLD*{GOLD_MULT} + SILVER) (semi-clean)")
    print("=" * 60)

    processed_mix_b2 = (processed_gold * GOLD_MULT) + processed_silver
    np.random.shuffle(processed_mix_b2)
    train_dataset_b2 = NERDataset(processed_mix_b2)

    cw_b2 = compute_class_weights(processed_mix_b2, num_labels=len(label_list), power=0.5)
    cw_b2 = torch.clamp(cw_b2, min=0.5, max=5.0)

    args_b2 = make_args(output_dir=ckpt_b2, lr=2e-5, epochs=2, warmup_ratio=0.05)
    model_b2 = init_model(ckpt_b1)

    trainer_b2 = WeightedLossTrainer(
        model=model_b2,
        args=args_b2,
        train_dataset=train_dataset_b2,
        eval_dataset=dev_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics_seqeval,
        class_weights=cw_b2,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1, early_stopping_threshold=0.0)],
    )

    trainer_b2.train()
    trainer_b2.save_model(ckpt_b2)
    tokenizer.save_pretrained(ckpt_b2)

    final_model_dir = ckpt_b2

print("\nFINAL MODEL DIR:", final_model_dir)



STAGE B1: 2026 GOLD (early stopping)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 368.22it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]              
BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-v1.1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.856897,0.383449,0.501860,0.434736
2,3.349703,0.433385,0.596024,0.768499,0.671362
3,0.923245,0.382971,0.660358,0.778834,0.714719
4,0.616515,0.375566,0.648847,0.802811,0.717664
5,0.495377,0.367870,0.672330,0.801571,0.731284
6,0.495377,0.377204,0.664184,0.811079,0.730318


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.37it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer


STAGE B2: (GOLD*5 + SILVER) (semi-clean)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 427.92it/s, Materializing param=classifier.weight]                                      


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.393356,0.351397,0.706729,0.811906,0.755675
2,0.274907,0.370793,0.718896,0.828855,0.769969


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.39it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer


FINAL MODEL DIR: models/bert_ner_2026_curriculum_B2_gold_plus_silver


## Load Model for Inference

In [42]:
inference_tokenizer = AutoTokenizer.from_pretrained(final_model_dir)
inference_model = AutoModelForTokenClassification.from_pretrained(final_model_dir)
inference_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inference_model.to(device)

print("✓ Inference model loaded")
print("✓ Device:", device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 331.26it/s, Materializing param=classifier.weight]                                      


✓ Inference model loaded
✓ Device: cuda


# INFERENCE

### Inference setup (load again + select device)


##  Two-pass threshold strategy (precision first, recall second)
 Instead of taking all decoded entities, we filter them by label-specific confidence thresholds.
 The idea:
 - Pass 1 uses strict thresholds to **keep only high-precision mentions**.
 - Pass 2 relaxes thresholds only for selected labels where recall is typically low.
#- Later **we merge the two passe**s while preventing overlaps with pass-1 outputs.

 This is a post-processing policy that can improve macro-F1 by balancing precision/recall per label.

In [43]:
# Pass 1: your best (high precision)
LABEL_THRESH_HIGH = {
  "DDF": 0.88,
  "bacteria": 0.84,
  "statistical technique": 0.91,
  "biomedical technique": 0.78,
  "gene": 0.68,
  "food": 0.60,
  "chemical": 0.72,
  "dietary supplement": 0.78,
  "drug": 0.80,
  "microbiome": 0.78,
  "anatomical location": 0.78,
  "human": 0.70,
  "animal": 0.70,
}

LABEL_THRESH_RECALL = {
    "food": 0.43,               # centro del range stabile 0.40–0.45
    "chemical": 0.65,           # best nella tua ricerca
    "bacteria": 0.80,           # best
    "dietary supplement": 0.72, # best (ma equivalente)
}
RECALL_LABELS = {"chemical", "food", "bacteria", "dietary supplement"}


DEFAULT_THRESH = 0.80

### Simple false-positive filters + label-specific postprocessing
This cell defines lightweight heuristics to remove obvious junk predictions:
- generic terms that are too unspecific (e.g., "microbes")
- markup fragments ("<...>")
- too short spans

It also adds a targeted postprocessing rule:
 - If something looks gene-like (e.g., IL-6, TNF-α, α-synuclein) but was predicted as "chemical",  remap it to "gene". This fixes a common confusion pattern that we observed.

In [44]:
import re
BAD_BACTERIA = {"bacteria", "micro", "microbes", "microorganisms", "genera", "taxa"}
BAD_CHEMICAL = {"metabolites", "neurotransmitters"}
BAD_DIETSUPP = {"nnss"}
BAD_MICROBIOME = {"micro", "microbiota", "gut"}
DIET_CONCEPT = {
    "diet", "ketogenic diet", "high-fat diet", "high fat diet",
    "high glycemic diet", "vegetarian diet", "balanced diet",
    "western diet", "mediterranean diet"
}
BAD_FOOD_EXACT = {
    "control", "ketogenic", "high-fat", "high fat", "high",
    "glycemic index", "lycemic index",
    "food", "ingested food"
}
def normalize_span(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def apply_simple_filters(entities):
    cleaned = []
    for e in entities:
        s = normalize_span(e["text_span"])

        # drop obvious HTML/markup garbage
        if "<" in s or ">" in s:
            continue

        # drop empty/very short spans
        if len(s) <= 1:
            continue

        # bacteria generic junk
        if e["label"] == "bacteria" and s in BAD_BACTERIA:
            continue
        # dietary supplement junk
        if e["label"] == "dietary supplement" and s in BAD_DIETSUPP:
            continue
        if e["label"] == "chemical" and s in BAD_CHEMICAL:
            continue
        if e["label"] == "microbiome" and s in BAD_MICROBIOME:
            continue
        if e["label"] == "food":
            if s in DIET_CONCEPT or s.endswith(" diet"):
                continue
            if s in BAD_FOOD_EXACT:
                continue
        cleaned.append(e)
    return cleaned

# -------------------------
# 3) Gene vs Chemical postprocess
# -------------------------
GENE_LIKE = re.compile(
    r"^(il-\d+|tnf(-?α)?|ifn(-?γ)?|tgf(-?β\d*)?|snca|park7|dj-1|mapt|apoe\d*|hla-[a-z0-9\*\:]+)$",
    re.IGNORECASE,
)
CHEM_LIKE = re.compile(
    r"(aβ|amyloid|scfa|gaba|succinate|butyrate|propionate|acetate|\b[a-z]+ate\b|\b[a-z]+acid\b|\(\d+\-\d+\))",
    re.IGNORECASE,
)

def postprocess_gene_vs_chemical(entities):
    for e in entities:
        s = normalize_span(e["text_span"])
        if e["label"] == "chemical" and GENE_LIKE.match(s):
            e["label"] = "gene"
        elif e["label"] == "gene" and CHEM_LIKE.search(s):
            e["label"] = "chemical"
    return entities

FOOD_ANCHORS = {"kefir", "yogurt", "milk", "cheese", "cookie", "lentil", "lentils", "buckwheat", "wheat", "rice", "tea", "coffee"}
SUPP_HARD = {"capsule", "tablet", "extract", "powder"}
SUPP_SOFT = {"probiotic", "probiotics", "prebiotic", "prebiotics", "synbiotic", "synbiotics", "supplement"}

def postprocess_food_vs_supp(entities):
    for e in entities:
        s = normalize_span(e["text_span"])

        if any(w in s for w in FOOD_ANCHORS):
            # se è un alimento riconoscibile, trattalo come food
            if e["label"] in {"dietary supplement", "food"}:
                e["label"] = "food"
            continue

        if any(w in s for w in SUPP_HARD):
            if e["label"] in {"dietary supplement", "food"}:
                e["label"] = "dietary supplement"
            continue

        if e["label"] == "food" and any(w in s for w in SUPP_SOFT):
            e["label"] = "dietary supplement"
    return entities



### Core predictor: decode BIO + compute entity confidence score
This is the main inference function that:
1) tokenizes text with offsets
2) runs the model to get logits -> softmax probabilities
3) converts per-token predictions to BIO labels
4) rebuilds entity spans by scanning tokens left-to-right

**Scoring:**
 - For each entity we compute a confidence score as the mean token probability
 over the entity span (B-tag prob for first token + I-tag prob for continuation tokens).

**BIO repair:**
- If we see I-X without an active entity, we start a new entity (treat as B-X).
- If we see I-X but we are currently inside Y, we close Y and start X.

 These rules make decoding more robust to occasional BIO inconsistencies

In [45]:
def predict_entities_with_scores(
    model,
    tokenizer,
    text: str,
    id2label: dict,
    label2id: dict,
    max_length: int = 512,
):
    """
    Returns list of entities with:
      start_idx (inclusive), end_idx (inclusive), label, text_span, score
    Score = mean token probability over the entity span.

    BIO repair:
      - I-X without an active entity => start new entity as X (treat as B-X)
      - I-X with different active label => close current and start new X
    """
    if not text:
        return []

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        return_offsets_mapping=True,
        max_length=max_length,
    )

    offsets = enc.pop("offset_mapping")[0].cpu().numpy()
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        out = model(**enc)
        logits = out.logits[0]  # [T, C]
        probs = torch.softmax(logits, dim=-1)  # [T, C]
        pred_ids = torch.argmax(probs, dim=-1).cpu().numpy()
        probs_cpu = probs.cpu().numpy()

    labels = [id2label[int(i)] for i in pred_ids]

    entities = []
    current = None

    def _start_entity(ent_label: str, s: int, e: int, t_idx: int):
        # e is exclusive in offsets; store inclusive in entity
        prob_idx = label2id.get(f"B-{ent_label}", None)
        token_prob = float(probs_cpu[t_idx, prob_idx]) if prob_idx is not None else float(probs_cpu[t_idx].max())
        return {
            "start_idx": s,
            "end_idx": e - 1,
            "label": ent_label,
            "text_span": text[s:e],
            "_token_probs": [token_prob],
        }

    def _extend_entity(ent: dict, e: int, t_idx: int):
        ent_label = ent["label"]
        prob_idx = label2id.get(f"I-{ent_label}", None)
        token_prob = float(probs_cpu[t_idx, prob_idx]) if prob_idx is not None else float(probs_cpu[t_idx].max())
        ent["end_idx"] = e - 1
        ent["text_span"] = text[ent["start_idx"]:e]
        ent["_token_probs"].append(token_prob)

    for t_idx, (lab, (s, e)) in enumerate(zip(labels, offsets)):
        s = int(s); e = int(e)

        # special tokens
        if s == 0 and e == 0:
            continue
        if e <= s:
            continue

        if lab.startswith("B-"):
            if current is not None:
                entities.append(current)
            ent_label = lab[2:]
            current = _start_entity(ent_label, s, e, t_idx)

        elif lab.startswith("I-"):
            ent_label = lab[2:]

            # --- BIO REPAIR ---
            if current is None:
                # I-X without B => start new X
                current = _start_entity(ent_label, s, e, t_idx)
                continue

            if ent_label != current["label"]:
                # I-X but current is Y => close Y and start X
                entities.append(current)
                current = _start_entity(ent_label, s, e, t_idx)
                continue

            # normal extend
            _extend_entity(current, e, t_idx)

        else:
            if current is not None:
                entities.append(current)
                current = None

    if current is not None:
        entities.append(current)

    # add score
    for ent in entities:
        probs_list = ent.pop("_token_probs", [])
        ent["score"] = float(np.mean(probs_list)) if probs_list else 0.0

    return entities



###  Threshold filtering (label -> threshold map)
 This helper keeps an entity only if:
- its computed score >= the threshold configured for its label
-  Labels not explicitly listed use `DEFAULT_THRESH`

In [46]:
def passes_threshold(e, label_thresh):
    thr = label_thresh.get(e["label"], DEFAULT_THRESH)
    s = normalize_span(e["text_span"])

    # food: se c'è "diet" alza la soglia (evita FP tipo "ketogenic diet")
    if e["label"] == "food" and ("diet" in s):
        thr = max(thr, 0.70)

    return e.get("score", 0.0) >= thr


def filter_by_threshold_with_map(entities, label_thresh):
    return [e for e in entities if passes_threshold(e, label_thresh)]


### Merge policy (no overlap with pass-1)
We merge two entity lists (pass-1 and pass-2) with a conservative rule:
- Keep all pass-1 (high precision)
- Add pass-2 entities only for selected labels (`RECALL_LABELS`)
- Add only if their character span does not overlap any already kept entity in the same segment
This avoids creating duplicated/competing spans that often hurt precision.

In [47]:
def span_iou(a, b):
    inter = max(0, min(a["end_idx"], b["end_idx"]) - max(a["start_idx"], b["start_idx"]) + 1)
    if inter == 0:
        return 0.0
    la = a["end_idx"] - a["start_idx"] + 1
    lb = b["end_idx"] - b["start_idx"] + 1
    return inter / (la + lb - inter)


def any_overlap(ent, kept, iou_thr=0.5):
    for k in kept:
        if k["location"] != ent["location"]:
            continue
        if ent["label"] == "food":
            # blocca solo se è quasi identico (doppione vero)
            if span_iou(ent, k) >= 0.85 and ent["label"] == k["label"]:
                return True
            continue

        if span_iou(ent, k) >= iou_thr:
            return True
    return False

def merge_two_pass(ents_high, ents_rec, recall_labels):
    kept = list(ents_high)
    for e in ents_rec:
        if e["label"] not in recall_labels:
            continue
        if not any_overlap(e, kept):
            kept.append(e)
    return kept


### Two-pass segment predictor + dev inference loop
 This function wraps the full inference pipeline for one text segment:
 - decode entities with scores
 - pass 1: strict thresholds + filters + postprocessing
 - pass 2: relaxed thresholds (selected labels) + filters + postprocessing
 - merge with "no overlap with pass-1" policy
 - remove the score field so the output matches submission schema

 Then we run it across all dev segments and aggregate entities back by PMID.

In [48]:
TRIM_CHARS = " \t\n\r.,;:()[]{}<>\"'"

def trim_entity_span(e, text):
    s = int(e["start_idx"])
    end = int(e["end_idx"])

    # safe bounds
    s = max(0, min(s, len(text)))
    end = max(0, min(end, len(text)-1))

    # trim left
    while s <= end and text[s] in TRIM_CHARS:
        s += 1
    # trim right
    while end >= s and text[end] in TRIM_CHARS:
        end -= 1

    if s <= end:
        e["start_idx"] = s
        e["end_idx"] = end
        e["text_span"] = text[s:end+1]
    return e


In [49]:
def predict_segment_entities_two_pass(model, tokenizer, text, location):
    ents_raw = predict_entities_with_scores(
        model=model,
        tokenizer=tokenizer,
        text=text,
        id2label=id2label,
        label2id=label2id,
        max_length=512,
    )
    ents_raw = [trim_entity_span(e, text) for e in ents_raw]

    # pass 1 (high precision)
    ents_high = filter_by_threshold_with_map(ents_raw, LABEL_THRESH_HIGH)
    ents_high = apply_simple_filters(ents_high)
    ents_high = postprocess_gene_vs_chemical(ents_high)
    ents_high = postprocess_food_vs_supp(ents_high)


    for e in ents_high:
        e["location"] = location

    # pass 2 (recall)
    ents_rec = filter_by_threshold_with_map(ents_raw, LABEL_THRESH_RECALL)
    ents_rec = apply_simple_filters(ents_rec)
    ents_rec = postprocess_gene_vs_chemical(ents_rec)
    ents_rec  = postprocess_food_vs_supp(ents_rec)
    for e in ents_rec:
        e["location"] = location

    merged = merge_two_pass(ents_high, ents_rec, recall_labels=RECALL_LABELS)

    # Remove score for submission compatibility
    for e in merged:
        e.pop("score", None)

    return merged

# -------------------------
# 8) Predict on dev set
# -------------------------
print("Running inference on dev set (two-pass)...")

predictions_two_pass = {}

for doc in tqdm(dev_documents, desc="Predicting (two-pass)"):
    pmid = doc["pmid"]
    location = doc["location"]
    text = doc["text"]

    ents = predict_segment_entities_two_pass(inference_model, inference_tokenizer, text, location)

    predictions_two_pass.setdefault(pmid, {"entities": []})
    predictions_two_pass[pmid]["entities"].extend(ents)

print(f"✓ Inference completed: {len(predictions_two_pass)} documents")
total_entities = sum(len(p["entities"]) for p in predictions_two_pass.values())
print(f"  Total entities predicted: {total_entities}")


Running inference on dev set (two-pass)...


Predicting (two-pass): 100%|██████████| 160/160 [00:04<00:00, 33.22it/s]

✓ Inference completed: 80 documents
  Total entities predicted: 2271


## Save Predictions

In [50]:
# Save predictions to file
output_path = "C:/Users/super/Documents/UniPd/ATA/GutBrainIE/src/predictions/bert_NER_twopass_stage_training.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(predictions_two_pass, f, ensure_ascii=False, indent=2)

print(f"Predictions saved to {output_path}")

Predictions saved to C:/Users/super/Documents/UniPd/ATA/GutBrainIE/src/predictions/bert_NER_twopass_stage_training.json


## Error inspection on predictions
 1) **Per-label** precision/recall/F1 using exact span match.
 2) Overlap-based **confusion matrix** (best IoU match) to see label confusions.
 3) **Boundary error report**: correct label but wrong offsets (plus "near misses" within ±k chars).
 4) Most frequent **false-positive** strings per label (helps refine filters).

These tools are meant for iterative improvement (thresholds, filters, span handling).

In [51]:
from collections import defaultdict
import pandas as pd
import re

# ----------------------------
# Helpers
# ----------------------------
def norm_span(s: str) -> str:
    """Normalize span text for pattern analysis (FP strings)."""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def ent_key(ent):
    # inclusive end_idx per your format
    return (int(ent["start_idx"]), int(ent["end_idx"]), str(ent["location"]), str(ent["label"]))

def ent_key_no_label(ent):
    return (int(ent["start_idx"]), int(ent["end_idx"]), str(ent["location"]))

def as_span(ent):
    # return (start, end_inclusive)
    return (int(ent["start_idx"]), int(ent["end_idx"]))

def overlap_len(a_start, a_end, b_start, b_end):
    # inclusive ends
    left = max(a_start, b_start)
    right = min(a_end, b_end)
    return max(0, right - left + 1)

def iou(a_start, a_end, b_start, b_end):
    inter = overlap_len(a_start, a_end, b_start, b_end)
    if inter == 0:
        return 0.0
    a_len = a_end - a_start + 1
    b_len = b_end - b_start + 1
    union = a_len + b_len - inter
    return inter / union

def build_index(entities):
    """
    Build location-based index for quick overlap checks.
    entities: list of dicts each with start_idx/end_idx/location/label/text_span
    """
    idx = defaultdict(list)  # loc -> list[(start,end,ent)]
    for e in entities:
        s, eend = as_span(e)
        loc = str(e["location"])
        idx[loc].append((s, eend, e))
    # sort by start for mild speed-up
    for loc in idx:
        idx[loc].sort(key=lambda x: x[0])
    return idx

def best_overlap_match(gold_ent, pred_candidates, min_iou=0.1):
    """
    Return best predicted entity overlapping the gold one, by IoU (ties by overlap length).
    pred_candidates: list[(start,end,ent)]
    """
    gs, ge = as_span(gold_ent)
    best = None
    best_iou = 0.0
    best_ol = 0

    for ps, pe, pent in pred_candidates:
        ol = overlap_len(gs, ge, ps, pe)
        if ol == 0:
            continue
        score = iou(gs, ge, ps, pe)
        if score < min_iou:
            continue
        if (score > best_iou) or (score == best_iou and ol > best_ol):
            best = pent
            best_iou = score
            best_ol = ol

    return best, best_iou, best_ol


# ----------------------------
# Flatten gold + pred
# ----------------------------
def flatten_gold(dev_data):
    gold = defaultdict(list)  # pmid -> list[ent]
    for pmid, article in dev_data.items():
        for e in article["entities"]:
            gold[pmid].append({
                "start_idx": int(e["start_idx"]),
                "end_idx": int(e["end_idx"]),
                "location": str(e["location"]),
                "label": str(e["label"]),
                "text_span": str(e.get("text_span", "")),
            })
    return gold

def flatten_pred(predictions):
    pred = defaultdict(list)
    for pmid, obj in predictions.items():
        for e in obj.get("entities", []):
            pred[pmid].append({
                "start_idx": int(e["start_idx"]),
                "end_idx": int(e["end_idx"]),
                "location": str(e["location"]),
                "label": str(e["label"]),
                "text_span": str(e.get("text_span", "")),
            })
    return pred


gold_by_pmid = flatten_gold(dev_data)
pred_by_pmid = flatten_pred(predictions_two_pass)

ALL_LABELS = sorted(set(
    [e["label"] for pmid in gold_by_pmid for e in gold_by_pmid[pmid]] +
    [e["label"] for pmid in pred_by_pmid for e in pred_by_pmid[pmid]]
))


# ----------------------------
# (1) Per-label Precision/Recall/F1
# ----------------------------
def per_label_prf(gold_by_pmid, pred_by_pmid, labels):
    gold_sets = {lab: set() for lab in labels}
    pred_sets = {lab: set() for lab in labels}

    for pmid, gold_ents in gold_by_pmid.items():
        for e in gold_ents:
            k = (pmid,) + ent_key(e)  # include pmid
            gold_sets[e["label"]].add(k)

    for pmid, pred_ents in pred_by_pmid.items():
        for e in pred_ents:
            k = (pmid,) + ent_key(e)
            pred_sets[e["label"]].add(k)

    rows = []
    for lab in labels:
        g = gold_sets[lab]
        p = pred_sets[lab]
        tp = len(g & p)
        fp = len(p - g)
        fn = len(g - p)

        prec = tp / (tp + fp + 1e-12)
        rec = tp / (tp + fn + 1e-12)
        f1 = 2 * prec * rec / (prec + rec + 1e-12)

        rows.append({
            "label": lab,
            "gold": len(g),
            "pred": len(p),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": prec,
            "recall": rec,
            "f1": f1,
        })

    df = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)
    return df

df_prf = per_label_prf(gold_by_pmid, pred_by_pmid, ALL_LABELS)
print("\n=== Per-label Precision / Recall / F1 ===")
print(df_prf.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


# ----------------------------
# (2) Confusion matrix (overlap-based)
# ----------------------------
def confusion_matrix_overlap(gold_by_pmid, pred_by_pmid, labels, min_iou=0.1):
    conf = pd.DataFrame(0, index=labels + ["<NONE>"], columns=labels + ["<NONE>"], dtype=int)

    for pmid in gold_by_pmid.keys():
        gold_ents = gold_by_pmid.get(pmid, [])
        pred_ents = pred_by_pmid.get(pmid, [])
        pred_idx = build_index(pred_ents)

        used_pred = set()  # track exact pred entities used in matches (by object id tuple)
        # Map gold -> best pred overlap
        for g in gold_ents:
            loc = g["location"]
            best, best_iou, _ = best_overlap_match(g, pred_idx.get(loc, []), min_iou=min_iou)
            g_lab = g["label"]

            if best is None:
                conf.loc[g_lab, "<NONE>"] += 1
            else:
                bkey = (best["start_idx"], best["end_idx"], best["location"], best["label"], best.get("text_span",""))
                used_pred.add(bkey)
                conf.loc[g_lab, best["label"]] += 1

        # Preds with no overlap-match to any gold count as <NONE> -> pred_label
        gold_idx = build_index(gold_ents)
        for p in pred_ents:
            pkey = (p["start_idx"], p["end_idx"], p["location"], p["label"], p.get("text_span",""))
            if pkey in used_pred:
                continue
            loc = p["location"]
            best_gold, _, _ = best_overlap_match(p, gold_idx.get(loc, []), min_iou=min_iou)
            if best_gold is None:
                conf.loc["<NONE>", p["label"]] += 1

    return conf

conf = confusion_matrix_overlap(gold_by_pmid, pred_by_pmid, ALL_LABELS, min_iou=0.1)
print("\n=== Confusion Matrix (rows=gold, cols=pred, overlap-based) ===")
# show top-left slice if huge
print(conf.to_string())


# ----------------------------
# (3) Boundary error rate (same label, overlap but offsets differ)
#     + "near miss" within +/- k chars
# ----------------------------
def boundary_report(gold_by_pmid, pred_by_pmid, labels, min_iou=0.1, near_k=3):
    # counts per label
    exact_tp = Counter()
    boundary_mismatch = Counter()
    near_miss = Counter()

    for pmid in gold_by_pmid.keys():
        gold_ents = gold_by_pmid.get(pmid, [])
        pred_ents = pred_by_pmid.get(pmid, [])
        pred_idx = build_index(pred_ents)

        # exact label+offset TP set for fast check
        pred_exact = set((pmid,) + ent_key(e) for e in pred_ents)

        for g in gold_ents:
            lab = g["label"]
            gk = (pmid,) + ent_key(g)

            if gk in pred_exact:
                exact_tp[lab] += 1
                continue

            # find best overlapping pred
            loc = g["location"]
            best, best_iou, _ = best_overlap_match(g, pred_idx.get(loc, []), min_iou=min_iou)
            if best is None:
                continue

            # boundary mismatch: same label but not exact offsets
            if best["label"] == lab:
                boundary_mismatch[lab] += 1

                # near miss: offsets close (start/end within +/- near_k)
                if (abs(best["start_idx"] - g["start_idx"]) <= near_k) and (abs(best["end_idx"] - g["end_idx"]) <= near_k):
                    near_miss[lab] += 1

    rows = []
    for lab in labels:
        tp = exact_tp[lab]
        bm = boundary_mismatch[lab]
        nm = near_miss[lab]
        denom = tp + bm
        rate = bm / (denom + 1e-12)  # among correct-label matches, how often boundaries differ
        rows.append({
            "label": lab,
            "exact_TP": tp,
            "boundary_mismatch_same_label": bm,
            "boundary_error_rate": rate,
            f"near_miss_within_±{near_k}": nm,
        })

    df = pd.DataFrame(rows).sort_values("boundary_error_rate", ascending=False).reset_index(drop=True)
    return df

df_boundary = boundary_report(gold_by_pmid, pred_by_pmid, ALL_LABELS, min_iou=0.1, near_k=3)
print("\n=== Boundary Errors (same label overlap but offsets differ) ===")
print(df_boundary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


# ----------------------------
# (4) FP patterns: top false-positive strings per label
# ----------------------------
def fp_patterns(gold_by_pmid, pred_by_pmid, top_n=15):
    # gold exact set per pmid for TP check
    gold_exact = set()
    for pmid, gold_ents in gold_by_pmid.items():
        for e in gold_ents:
            gold_exact.add((pmid,) + ent_key(e))

    fp_by_label = defaultdict(Counter)

    for pmid, pred_ents in pred_by_pmid.items():
        for e in pred_ents:
            pk = (pmid,) + ent_key(e)
            if pk in gold_exact:
                continue  # true positive
            # false positive
            lab = e["label"]
            fp_by_label[lab][norm_span(e.get("text_span", ""))] += 1

    # Pretty print
    for lab, counter in sorted(fp_by_label.items(), key=lambda x: sum(x[1].values()), reverse=True):
        print(f"\n=== Top FP strings for label: {lab} (total FP={sum(counter.values())}) ===")
        for span, c in counter.most_common(top_n):
            if span == "":
                span = "<EMPTY>"
            print(f"{c:>4}  {span}")

fp_patterns(gold_by_pmid, pred_by_pmid, top_n=15)



=== Per-label Precision / Recall / F1 ===
                label  gold  pred  tp  fp  fn  precision  recall     f1
                human   192   179 170   9  22     0.9497  0.8854 0.9164
           microbiome   231   215 204  11  27     0.9488  0.8831 0.9148
                  DDF   793   720 663  57 130     0.9208  0.8361 0.8764
                 drug    75    74  64  10  11     0.8649  0.8533 0.8591
               animal   152   144 123  21  29     0.8542  0.8092 0.8311
  anatomical location   169   162 130  32  39     0.8025  0.7692 0.7855
statistical technique    35    34  25   9  10     0.7353  0.7143 0.7246
             bacteria   183   151 121  30  62     0.8013  0.6612 0.7246
   dietary supplement    64    61  44  17  20     0.7213  0.6875 0.7040
             chemical   366   301 223  78 143     0.7409  0.6093 0.6687
                 gene    63    74  45  29  18     0.6081  0.7143 0.6569
 biomedical technique   139   126  87  39  52     0.6905  0.6259 0.6566
                 food

## EVALUATION
Official-style evaluation (macro + micro)

In [52]:
# Load evaluation functions from evaluate.py concepts
def remove_duplicated_entities(predictions):
    """Remove duplicated entities from predictions."""
    removed_count = 0
    for pmid in list(predictions.keys()):
        seen = set()
        deduped = []
        for ent in predictions[pmid]["entities"]:
            #key = (ent["start_idx"], ent["end_idx"], ent["location"])
            key = (ent["start_idx"], ent["end_idx"], ent["location"], ent["label"])

            if key not in seen:
                seen.add(key)
                deduped.append(ent)
            else:
                removed_count += 1
        predictions[pmid]["entities"] = deduped
    
    if removed_count > 0:
        print(f"Removed {removed_count} duplicated entities from predictions")

def remove_overlapping_entities_eval(predictions):
    """Remove overlapping entities, keeping longest spans."""
    removed_count = 0

    for pmid in list(predictions.keys()):
        original_len = len(predictions[pmid]['entities'])
        
        groups = {'title': [], 'abstract': []}
        for ent in predictions[pmid]['entities']:
            loc = ent["location"]
            groups[loc].append(ent)

        keepers = set()
        for loc in groups:
            group = groups[loc]
            group = sorted(group, key=lambda e: e["start_idx"])

            clusters = []
            cluster = []
            current_end = None

            for ent in group:
                if not cluster:
                    cluster = [ent]
                    current_end = ent["end_idx"]
                else:
                    if ent["start_idx"] < current_end:
                        cluster.append(ent)
                        if ent["end_idx"] > current_end:
                            current_end = ent["end_idx"]
                    else:
                        clusters.append(cluster)
                        cluster = [ent]
                        current_end = ent["end_idx"]
            if cluster:
                clusters.append(cluster)

            for clust in clusters:
                longest = clust[0]
                max_len = longest["end_idx"] - longest["start_idx"]
                for ent in clust[1:]:
                    length = ent["end_idx"] - ent["start_idx"]
                    if length > max_len:
                        longest = ent
                        max_len = length
                keepers.add((longest["start_idx"],
                             longest["end_idx"],
                             longest["location"]))

        deduped = []
        for ent in predictions[pmid]['entities']:
            key = (ent["start_idx"], ent["end_idx"], ent["location"])
            if key in keepers:
                deduped.append(ent)
                keepers.remove(key)

        predictions[pmid]["entities"] = deduped
        removed_count += (original_len - len(deduped))

    if removed_count > 0:
        print(f"Removed {removed_count} overlapping entities")

print("✓ Evaluation helper functions defined")

✓ Evaluation helper functions defined


In [53]:
def evaluate_ner(predictions, ground_truth):
    """Evaluate NER predictions against ground truth."""
    # Remove duplicated and overlapping entities
    remove_duplicated_entities(predictions)
    remove_overlapping_entities_eval(predictions)
    
    LEGAL_ENTITY_LABELS = [
        "anatomical location", "animal", "bacteria", "biomedical technique",
        "chemical", "DDF", "dietary supplement", "drug", "food", "gene",
        "human", "microbiome", "statistical technique"
    ]
    
    ground_truth_NER = dict()
    count_annotated_entities_per_label = {}
    
    for pmid, article in ground_truth.items():
        if pmid not in ground_truth_NER:
            ground_truth_NER[pmid] = []
        for entity in article['entities']:
            start_idx = int(entity["start_idx"])
            end_idx = int(entity["end_idx"])
            location = str(entity["location"])
            text_span = str(entity["text_span"])
            label = str(entity["label"]) 
            
            entry = (start_idx, end_idx, location, text_span, label)
            ground_truth_NER[pmid].append(entry)
            
            if label not in count_annotated_entities_per_label:
                count_annotated_entities_per_label[label] = 0
            count_annotated_entities_per_label[label] += 1

    count_predicted_entities_per_label = {label: 0 for label in list(count_annotated_entities_per_label.keys())}
    count_true_positives_per_label = {label: 0 for label in list(count_annotated_entities_per_label.keys())}

    for pmid in predictions.keys():
        entities = predictions[pmid]['entities']
        
        for entity in entities:
            start_idx = int(entity["start_idx"])
            end_idx = int(entity["end_idx"])
            location = str(entity["location"])
            text_span = str(entity["text_span"])
            label = str(entity["label"]) 
            
            if label not in LEGAL_ENTITY_LABELS:
                continue

            if label in count_predicted_entities_per_label:
                count_predicted_entities_per_label[label] += 1

            entry = (start_idx, end_idx, location, text_span, label)
            if pmid in ground_truth_NER and entry in ground_truth_NER[pmid]:
                count_true_positives_per_label[label] += 1

    count_annotated_entities = sum(count_annotated_entities_per_label.values())
    count_predicted_entities = sum(count_predicted_entities_per_label.values())
    count_true_positives = sum(count_true_positives_per_label.values())

    micro_precision = count_true_positives / (count_predicted_entities + 1e-10)
    micro_recall = count_true_positives / (count_annotated_entities + 1e-10)
    micro_f1 = 2 * ((micro_precision * micro_recall) / (micro_precision + micro_recall + 1e-10))

    precision, recall, f1 = 0, 0, 0
    n = len(count_annotated_entities_per_label)
    for label in count_annotated_entities_per_label.keys():
        current_precision = count_true_positives_per_label[label] / (count_predicted_entities_per_label[label] + 1e-10) 
        current_recall = count_true_positives_per_label[label] / (count_annotated_entities_per_label[label] + 1e-10) 
        
        precision += current_precision
        recall += current_recall
        f1 += 2 * ((current_precision * current_recall) / (current_precision + current_recall + 1e-10))
    
    precision = precision / n
    recall = recall / n
    f1 = f1 / n

    return precision, recall, f1, micro_precision, micro_recall, micro_f1


# Evaluate
precision, recall, f1, micro_precision, micro_recall, micro_f1 = evaluate_ner(predictions_two_pass, dev_data)

print("="*60)
print("BERT NER BASELINE RESULTS")
print("="*60)
print("\nMacro-averaged Metrics:")
print(f"  Macro-Precision: {precision:.4f}")
print(f"  Macro-Recall:    {recall:.4f}")
print(f"  Macro-F1 Score:  {f1:.4f}")

print("\nMicro-averaged Metrics:")
print(f"  Micro-Precision: {micro_precision:.4f}")
print(f"  Micro-Recall:    {micro_recall:.4f}")
print(f"  Micro-F1 Score:  {micro_f1:.4f}")
print("="*60)

BERT NER BASELINE RESULTS

Macro-averaged Metrics:
  Macro-Precision: 0.8132
  Macro-Recall:    0.7326
  Macro-F1 Score:  0.7652

Micro-averaged Metrics:
  Micro-Precision: 0.8485
  Micro-Recall:    0.7644
  Micro-F1 Score:  0.8043


## Analysis: Entity Distribution by Label
- how many gold mentions exist in dev
- how many mentions your system predicts

 This helps spot systematic under/over-prediction:
 - predicted << gold => recall bottleneck for that label
- predicted >> gold => precision bottleneck for that label

In [54]:
from collections import Counter

# Count entities by label in predictions
pred_label_counts = Counter()
for pmid, pred in predictions_two_pass.items():
    for entity in pred['entities']:
        pred_label_counts[entity['label']] += 1

# Count entities by label in gold standard
gold_label_counts = Counter()
for pmid, article in dev_data.items():
    for entity in article['entities']:
        gold_label_counts[entity['label']] += 1

print("Entity Distribution by Label:")
print("="*60)
print(f"{'Label':<25} {'Gold':<10} {'Predicted':<10}")
print("-"*60)

all_labels = set(gold_label_counts.keys()) | set(pred_label_counts.keys())
for label in sorted(all_labels):
    print(f"{label:<25} {gold_label_counts[label]:<10} {pred_label_counts[label]:<10}")

print("-"*60)
print(f"{'TOTAL':<25} {sum(gold_label_counts.values()):<10} {sum(pred_label_counts.values()):<10}")

Entity Distribution by Label:
Label                     Gold       Predicted 
------------------------------------------------------------
DDF                       793        720       
anatomical location       169        162       
animal                    152        144       
bacteria                  183        151       
biomedical technique      139        126       
chemical                  366        301       
dietary supplement        64         61        
drug                      75         74        
food                      59         30        
gene                      63         74        
human                     192        179       
microbiome                231        215       
statistical technique     35         34        
------------------------------------------------------------
TOTAL                     2521       2271      
